<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/CH32_RAG_Query_Routing_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **RAG Query Routing**

> **1. Logical Routing**

> **2. Semantic Routing**

In [1]:
pip install langchain langchain-google-genai langchain-community langchain-core faiss-cpu -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.


In [3]:
# @title **Getting Set Up with LangChain**
import os
from google.colab import userdata
YOUR_API_KEY = userdata.get('Gemini-API-Key')

os.environ["GOOGLE_API_KEY"] = YOUR_API_KEY

In [ ]:
# @title 공통 준비 코드
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.load import dumps, loads

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

docs = [
    Document(page_content="RAG는 검색된 문서를 LLM의 컨텍스트에 넣어 답변 품질을 높이는 기법이다."),
    Document(page_content="FAISS는 Meta가 만든 고속 벡터 유사도 검색 라이브러리이다."),
    Document(page_content="Transformer는 self-attention 기반의 신경망 구조이며, 모든 토큰 쌍의 관계를 직접 계산해 장거리 의존성을 잘 처리한다."),
    Document(page_content="RAG 성능은 청킹 전략, 리랭킹, 쿼리 변환 등으로 개선할 수 있다."),
]

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

answer_prompt = ChatPromptTemplate.from_template(
    "다음 컨텍스트만 사용해 질문에 답하세요.\n\n컨텍스트:\n{context}\n\n질문: {question}"
)
answer_chain = answer_prompt | llm | StrOutputParser()

In [8]:
# @title 공통 준비 코드
from langchain_google_genai import ChatGoogleGenerativeAI

### **1. Logical Routing (LLM 기반 구조화된 출력 분기)**

LLM의 추론 능력을 활용하여 질문 의도에 맞게 분기

In [9]:
# @title **1. Logical Routing**

from pydantic import BaseModel, Field
from typing import Literal

# 라우팅 결과 스키마 정의
class RouteQuery(BaseModel):
    destination: Literal["python_db", "math_db"] = Field(
        description="질문의 내용에 따라 파이썬 관련은 'python_db', 수학 관련은 'math_db'로 선택"
    )

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

# Structured Output을 활용한 라우터 생성
structured_llm = llm.with_structured_output(RouteQuery)

# 실행 예시
query = "파이썬에서 리스트를 정렬하는 방법은?"
result = structured_llm.invoke(query)
print(f"선택된 경로: {result.destination}")  # 출력: python_db

선택된 경로: python_db


### **Semantic Routing (임베딩 유사도 기반 분기)**

LLM 생성 과정 없이 질문의 임베딩과 샘플 문장의 유사도를 기반으로 빠르게 분기

In [12]:
pip install docarray -q

In [13]:
# @title **2. Semantic Routing (임베딩 유사도 기반 분기)**
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.documents import Document

# **로컬 HuggingFace 임베딩 모델 소환**
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 대표 샘플 문장(예시 프롬프트) 사전 저장
routes = [
    Document(page_content="파이썬 코드 작성법, 문법, 프로그래밍 질문", metadata={"destination": "python_db"}),
    Document(page_content="수학 공식, 계산, 미분 적분 질문", metadata={"destination": "math_db"})
]

# 메모리 기반 벡터 스토어 생성
vectorstore = DocArrayInMemorySearch.from_documents(routes, embeddings)

# 실행 예시: 질문과 가장 유사한 경로 찾기
query = "파이썬에서 response 받는 코드 작성해줘"
matched_doc = vectorstore.similarity_search(query, k=1)[0]
print(f"선택된 경로: {matched_doc.metadata['destination']}")  # 출력: python_db

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

선택된 경로: python_db
